In [28]:
import os
from bs4 import BeautifulSoup

folder_path = r"E:\Github\uit_chatbot\graph\data\BoPhapDienDienTu\demuc"

html_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.html', '.htm'))]

all_dieu_texts = []

for html_file in html_files:
    file_path = os.path.join(folder_path, html_file)
    with open(file_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')

        # Chỉ lấy <p class='pDieu'>, đây mới là các "Điều" thực sự
        dieu_tags = soup.find_all('p', class_='pDieu')
        for tag in dieu_tags:
            text = tag.get_text(strip=True)
            if text.startswith("Điều"):
                all_dieu_texts.append(text)

In [7]:
# In kết quả
for idx, dieu in enumerate(all_dieu_texts, 1):
    print(f"{idx}. {dieu}")

1. Điều 36.3.LQ.1. Thanh niên
2. Điều 36.3.LQ.2. Phạm vi điều chỉnh
3. Điều 36.3.LQ.3. Đối tượng áp dụng
4. Điều 36.3.NĐ.1.1. Phạm vi điều chỉnh
5. Điều 36.3.NĐ.1.2. Đối tượng áp dụng
6. Điều 36.3.NĐ.2.1. Phạm vi điều chỉnh
7. Điều 36.3.NĐ.2.2. Đối tượng áp dụng
8. Điều 36.3.NĐ.3.1. Phạm vi điều chỉnh
9. Điều 36.3.NĐ.3.2. Đối tượng áp dụng
10. Điều 36.3.TL.1.1. Phạm vi và đối tượng áp dụng


In [8]:
print(len(all_dieu_texts))

72633


In [ ]:
import os
import re
from bs4 import BeautifulSoup

folder_path = r"E:\Github\uit_chatbot\graph\data\BoPhapDienDienTu\demuc"

regex_demuc = re.compile(r"Đề mục\s+\d+(\.\d+)*")   # bắt Đề mục 45 hoặc 45.5 hoặc 45.5.2

result = {}

html_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.html', '.htm'))]

for html_file in html_files:
    file_path = os.path.join(folder_path, html_file)

    with open(file_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    # tìm h3 vì Đề mục nằm trong h3
    h3 = soup.find("h3")
    if not h3:
        print(f"⚠ Không thấy <h3> trong {html_file}")
        continue

    text = h3.get_text(separator=" ", strip=True)

    # tìm Đề mục trong nội dung h3
    match = regex_demuc.search(text)
    if match:
        demuc = match.group()
        result[html_file] = demuc
    else:
        print(f"Không tìm thấy Đề mục trong {html_file}")

In [32]:
import csv

def parse_demuc(demuc_text):
    # demuc_text: "Đề mục 45.5"
    number_part = demuc_text.replace("Đề mục", "").strip()
    return [int(x) for x in number_part.split(".")]
sorted_items = sorted(result.items(), key=lambda item: parse_demuc(item[1]))

print("\nDanh sách Đề mục:")
# sort bằng key = parse_demuc(demuc)
for file, demuc in sorted_items:
    print(f"{file} → {demuc}")

# Save to CSV
OUTPUT_CSV = "demuc_list.csv"
with open(OUTPUT_CSV, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["demuc", "file"])  # header

    for file, demuc in sorted_items:
        writer.writerow([demuc, file])

print(f"Saved sorted Đề mục list to {OUTPUT_CSV}")


Danh sách Đề mục:
55323c64-e78f-4537-afcd-6a3c2af3c71d.html → Đề mục 1.1
84a4b90e-6b07-41ca-919d-759cfb657f3f.html → Đề mục 1.2
68079655-45cb-43a1-8c03-6603a7c52ec6.html → Đề mục 1.3
8b06986b-89cf-4a8d-8a2f-e14ff9e3123e.html → Đề mục 1.4
e499d586-a641-4a44-8591-7814730dc583.html → Đề mục 1.5
18b6dee2-0ff9-4cb7-b2b4-fb6ee352013e.html → Đề mục 1.6
7ca2c60d-7e8f-4604-b622-7d297acd506f.html → Đề mục 1.7
74d84211-de1e-44b4-b2c2-976d87c3190b.html → Đề mục 1.8
53892a89-6ea2-44c2-8f84-840fab0e8c47.html → Đề mục 1.9
e071327d-c715-4ad2-9dd7-e8c2aa4e912a.html → Đề mục 1.10
1fd42d83-9d78-4dd4-b6b6-73e9bd3472e1.html → Đề mục 1.11
f0f0c740-cb1f-4013-b5bc-9e14672d586d.html → Đề mục 1.12
7fe4f772-662c-4807-b5b3-d259807e94a4.html → Đề mục 1.13
c8ba5bc7-224a-4f31-8d92-567613f68ca9.html → Đề mục 2.1
583b913f-4d4b-46da-911c-112342c6ea49.html → Đề mục 2.2
4b27a678-6cea-4635-9bf3-4f3655ca74af.html → Đề mục 2.3
66c59035-9acd-430a-affd-86563bc7ce77.html → Đề mục 3.1
82e15104-9b5b-4b48-94f2-4ab37fc8e8a1.html 

In [4]:
import os
import csv
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import urljoin

# Paths
folder_path = r"E:\Github\uit_chatbot\graph\data\BoPhapDienDienTu\demuc"
csv_path = r"E:\Github\uit_chatbot\graph\data\BoPhapDienDienTu\demuc_list.csv"
download_dir = r"E:\Github\uit_chatbot\graph\data\BoPhapDienDienTu\downloads"
os.makedirs(download_dir, exist_ok=True)

# Target Đề mục
target_demucs = {"Đề mục 9.1", "Đề mục 9.2"}

# HTTP session
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "*/*",
})

# Load CSV
demuc_files = []
with open(csv_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        demuc_files.append((row["demuc"], row["file"]))

# Extract VBPL links
unique_links = set()
for demuc, html_file in demuc_files:
    if demuc not in target_demucs:
        continue
    file_path = os.path.join(folder_path, html_file)
    if not os.path.exists(file_path):
        continue
    with open(file_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if "vbpl.vn" in href:
                clean = href.split("#")[0].strip()
                clean = urljoin("https://vbpl.vn", clean)  # ensures absolute URL
                unique_links.add(clean)

print("Found VBPL links:", len(unique_links))

# Helper: parse javascript downloadfile
def convert_js_download(js_text):
    match = re.search(r"downloadfile\('([^']+)','([^']+)'\)", js_text)
    if not match:
        return None
    filename = match.group(1)
    path = match.group(2)
    url = urljoin("https://vbpl.vn", path)
    return url, filename

# Helper: download file
def download_file(url, filename):
    path = os.path.join(download_dir, filename)
    if os.path.exists(path):
        print("Already exists:", filename)
        return
    try:
        r = session.get(url, stream=True)
        if r.status_code == 200:
            with open(path, "wb") as f:
                for chunk in r.iter_content(1024):
                    f.write(chunk)
            print("Downloaded:", filename)
        else:
            print("Failed to download:", url, r.status_code)
    except Exception as e:
        print("Error downloading", url, e)

# Main loop
for link in unique_links:
    if link.endswith("ItemID=") or "ItemID=" not in link:
        print(f"Skipping invalid link: {link}")
        continue

    print("\nVisiting:", link)
    try:
        r = session.get(link)
        if r.status_code != 200:
            print("Cannot open page:", r.status_code)
            continue
    except Exception as e:
        print("Error:", e)
        continue

    soup = BeautifulSoup(r.text, "html.parser")

    # 1️⃣ vbFile DOC under "File đính kèm"
    vbfile_div = soup.find("div", class_="vbFile")
    doc_downloaded = False
    if vbfile_div:
        li_file = None
        for li in vbfile_div.find_all("li"):
            if "File đính kèm" in li.get_text():
                li_file = li
                break
        if li_file:
            # Try DOC first
            for a in li_file.find_all("a", href=True):
                href = a["href"]
                if "downloadfile" in href and "doc" in href:
                    result = convert_js_download(href)
                    if result:
                        url, filename = result
                        download_file(url, filename)
                        doc_downloaded = True

            # If no DOC, download PDFs
            if not doc_downloaded:
                for a in li_file.find_all("a", href=True):
                    href = a["href"]
                    if "downloadfile" in href and "pdf" in href:
                        result = convert_js_download(href)
                        if result:
                            url, filename = result
                            download_file(url, filename)

    # 2️⃣ vbProperties — download all files in <object data> or <a>
    vbprop_div = soup.find("div", class_="vbProperties")
if vbprop_div:
    # Handle <object> tags
    for obj in vbprop_div.find_all("object", data=True):
        data_url = obj["data"]
        url = urljoin("https://vbpl.vn", data_url)

        # Extract filename, removing query parameters
        filename = os.path.basename(url.split('?')[0])

        # If no valid filename, skip
        if not filename or filename == '':
            print(f"Skipping invalid object URL: {url}")
            continue

        download_file(url, filename)

    # Handle <a> tags
    for a in vbprop_div.find_all("a", href=True):
        href = a["href"]

        # Skip javascript links or empty hrefs
        if href.startswith('javascript:') or not href.strip():
            continue

        url = urljoin("https://vbpl.vn", href)

        # Extract filename, removing query parameters
        filename = os.path.basename(url.split('?')[0])

        # If no valid filename, skip
        if not filename or filename == '':
            print(f"Skipping invalid link URL: {url}")
            continue

        download_file(url, filename)

print("\nDone downloading files!")

Found VBPL links: 35

Visiting: http://vbpl.vn/TW/Pages/vbpq-toanvan.aspx?ItemID=139903

Visiting: http://vbpl.vn/TW/Pages/vbpq-toanvan.aspx?ItemID=36138
Downloaded: 43.2014.ND.CP.doc
Skipping invalid link: http://vbpl.vn/TW/Pages/vbpq-toanvan.aspx?ItemID=

Visiting: http://vbpl.vn/TW/Pages/vbpq-toanvan.aspx?ItemID=46751
Downloaded: 68.2014.QH13.doc

Visiting: http://vbpl.vn/TW/Pages/vbpq-toanvan.aspx?ItemID=27844
Downloaded: 01.2012.TTLT.TANDTC.VKSNDTC.BTP.doc
Downloaded: Bieu mau.doc

Visiting: http://vbpl.vn/TW/Pages/vbpq-toanvan.aspx?ItemID=165913
Downloaded: 20_2023_QH15_513347.doc

Visiting: http://vbpl.vn/TW/Pages/vbpq-toanvan.aspx?ItemID=146467

Visiting: http://vbpl.vn/TW/Pages/vbpq-toanvan.aspx?ItemID=38405
Downloaded: 23.2014.TT.BTNMT.doc

Visiting: http://vbpl.vn/TW/Pages/vbpq-toanvan.aspx?ItemID=76318
Downloaded: 68.2015.ND.CP.doc

Visiting: http://vbpl.vn/TW/Pages/vbpq-toanvan.aspx?ItemID=132336
Downloaded: NĐ.159.2018.NĐ.CP.doc

Visiting: http://vbpl.vn/TW/Pages/vbpq-toa

## Testing crawling Luat Dat Dai

In [23]:
import time
import csv
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://chinhphu.vn/he-thong-van-ban?classid=1&mode=1"
CATEGORY_VALUE = "21"
DELAY = 2

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
})

def get_hidden_fields(soup):
    """Extract all hidden form fields including __VIEWSTATE, __EVENTVALIDATION, etc."""
    fields = {}
    for tag in soup.find_all("input", type="hidden"):
        name = tag.get("name")
        if name:
            fields[name] = tag.get("value", "")
    return fields

def extract_documents(soup):
    documents = []
    table = soup.find("table", id="ctrl_191017_163_grvDocument")
    if not table:
        return documents

    all_rows = table.find_all("tr")
    rows = all_rows[1:-1]

    for row in rows:
        tds = row.find_all("td")
        if len(tds) < 3:
            continue

        code_span = tds[0].find("span", class_="code")
        code = code_span.get_text(strip=True) if code_span else ""

        date_span = tds[1].find("span", class_="issued-date")
        issued_date = date_span.get_text(strip=True) if date_span else ""

        summary_span = tds[2].find("span", class_="substract")
        summary = summary_span.get_text(strip=True) if summary_span else ""

        # --- Find ALL PDF links in this row ---
        pdf_links = []
        files_div = tds[2].find("div", class_="bl-doc-files")
        if files_div:
            all_links = files_div.find_all("a", href=True)
            for a in all_links:
                href = a["href"]
                if href.lower().endswith(".pdf"):
                    # Fix relative links
                    if href.startswith("/"):
                        href = "https://chinhphu.vn" + href
                    pdf_links.append(href)

        # If there are multiple PDFs, create separate entries for each
        if pdf_links:
            for i, pdf_link in enumerate(pdf_links):
                # Add suffix to code if there are multiple PDFs
                doc_code = code if len(pdf_links) == 1 else f"{code} (file {i+1})"
                documents.append({
                    "code": doc_code,
                    "date": issued_date,
                    "summary": summary,
                    "pdf": pdf_link
                })
        else:
            # No PDF found, still add the document with empty PDF field
            documents.append({
                "code": code,
                "date": issued_date,
                "summary": summary,
                "pdf": ""
            })

    return documents

def get_current_page_number(soup):
    """Extract current page number from the pagination (last row of table)"""
    table = soup.find("table", id="ctrl_191017_163_grvDocument")
    if not table:
        return 1

    all_rows = table.find_all("tr")
    if len(all_rows) < 2:
        return 1

    # Pagination is in the last row
    pagination_row = all_rows[-1]

    # Current page is in a <span> tag (not a link)
    current_span = pagination_row.find("span")
    if current_span:
        try:
            return int(current_span.get_text(strip=True))
        except:
            return 1
    return 1

def get_next_page_target(soup, current_page):
    """
    Get the next page to navigate to. Returns (event_argument, description) or (None, None).
    """
    table = soup.find("table", id="ctrl_191017_163_grvDocument")
    if not table:
        return None, None

    all_rows = table.find_all("tr")
    if len(all_rows) < 2:
        return None, None

    # Pagination is in the last row
    pagination_row = all_rows[-1]
    all_tds = pagination_row.find_all("td")

    # Find which td contains the current page span
    current_page_td_index = None
    for i, td in enumerate(all_tds):
        span = td.find("span")
        if span:
            try:
                if int(span.get_text(strip=True)) == current_page:
                    current_page_td_index = i
                    break
            except:
                pass

    # Strategy 1: Look for direct next page number (current + 1) AFTER current page
    next_page_num = current_page + 1
    if current_page_td_index is not None:
        for i in range(current_page_td_index + 1, len(all_tds)):
            td = all_tds[i]
            link = td.find("a")
            if link and link.get_text(strip=True) == str(next_page_num):
                href = link.get("href", "")
                if "Page$" in href:
                    page_arg = href.split("Page$")[1].rstrip("')")
                    return f"Page${page_arg}", f"Page {next_page_num}"

    # Strategy 2: Look for "..." that comes AFTER current page (forward ellipsis)
    if current_page_td_index is not None:
        for i in range(current_page_td_index + 1, len(all_tds)):
            td = all_tds[i]
            link = td.find("a")
            if link and link.get_text(strip=True) == "...":
                href = link.get("href", "")
                if "Page$" in href:
                    page_arg = href.split("Page$")[1].rstrip("')")
                    # Extract the page number from the argument
                    try:
                        target_page = int(page_arg)
                        # Only use this if it's forward (greater than current)
                        if target_page > current_page:
                            return f"Page${page_arg}", f"Next batch (page {target_page})"
                    except:
                        pass

    # Strategy 3: Look for forward arrow icon (last page)
    for td in all_tds:
        link = td.find("a")
        if link:
            if "fa-step-forward" in str(link):
                href = link.get("href", "")
                if "Page$Last" in href:
                    return "Page$Last", "Last page"

    return None, None


# Step 1: Initial GET
print("Loading initial page...")
response = session.get(BASE_URL)
soup = BeautifulSoup(response.text, "html.parser")

# Step 2: Submit search form
print("Submitting search form...")
hidden = get_hidden_fields(soup)

post_data = {
    "__EVENTTARGET": "",
    "__EVENTARGUMENT": "",
    "ctrl_191017_163$drdDocCategory": CATEGORY_VALUE,
    "ctrl_191017_163$drdRecordPerPage": "500",
    "ctrl_191017_163$btnSearch": "Tìm kiếm"
}
for key, value in hidden.items():
    if key not in post_data:
        post_data[key] = value

response = session.post(BASE_URL, data=post_data)
soup = BeautifulSoup(response.text, "html.parser")

all_documents = []
seen_docs = set()
iteration = 0
max_iterations = 50  # Safety limit

while iteration < max_iterations:
    iteration += 1
    current_page = get_current_page_number(soup)
    print(f"\n=== Iteration {iteration}: Page {current_page} ===")

    # Extract documents
    page_docs = extract_documents(soup)

    if not page_docs:
        print("No documents found. Stopping.")
        break

    # Show first document as verification
    if page_docs:
        print(f"  First doc: {page_docs[0]['code']}")

    # Check for duplicates
    new_count = 0
    for doc in page_docs:
        # Use code + pdf as unique identifier (in case same code has multiple files)
        doc_id = (doc['code'], doc['pdf'])
        if doc_id not in seen_docs:
            seen_docs.add(doc_id)
            all_documents.append(doc)
            new_count += 1

    print(f"  Found {new_count} new documents (total: {len(all_documents)})")

    if new_count == 0:
        print("All documents are duplicates - stopping.")
        break

    # Get next page target
    event_arg, description = get_next_page_target(soup, current_page)

    if not event_arg:
        print("No more pages available - stopping.")
        break

    print(f"  Navigating to: {description} (arg: {event_arg})")

    # Wait before next request
    time.sleep(DELAY)

    # Get fresh hidden fields
    hidden = get_hidden_fields(soup)

    # Build POST data for pagination
    post_data = {}

    # First, add all hidden fields
    for key, value in hidden.items():
        post_data[key] = value

    # Then set the postback parameters
    post_data["__EVENTTARGET"] = "ctrl_191017_163$grvDocument"
    post_data["__EVENTARGUMENT"] = event_arg

    # Keep the dropdown values
    post_data["ctrl_191017_163$drdDocCategory"] = CATEGORY_VALUE
    post_data["ctrl_191017_163$drdRecordPerPage"] = "500"

    # Make the request
    response = session.post(BASE_URL, data=post_data)

    if response.status_code != 200:
        print(f"  Error: Got status code {response.status_code}")
        break

    soup = BeautifulSoup(response.text, "html.parser")

    # Verify we moved to a different page
    new_page = get_current_page_number(soup)
    if new_page == current_page:
        print(f"  WARNING: Still on page {current_page}!")
        break

# Save results
print(f"\n{'='*50}")
print("Saving results...")
with open("documents.csv", mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["code", "date", "summary", "pdf"]
    )
    writer.writeheader()
    writer.writerows(all_documents)

print(f"✓ Successfully saved {len(all_documents)} unique documents to documents.csv")

Loading initial page...
Submitting search form...

=== Iteration 1: Page 1 ===
  First doc: 292/2025/NĐ-CP
  Found 55 new documents (total: 55)
  Navigating to: Page 2 (arg: Page$2)

=== Iteration 2: Page 2 ===
  First doc: 104/2024/NĐ-CP
  Found 54 new documents (total: 109)
  Navigating to: Page 3 (arg: Page$3)

=== Iteration 3: Page 3 ===
  First doc: 44/2022/QĐ-UBND
  Found 52 new documents (total: 161)
  Navigating to: Page 4 (arg: Page$4)

=== Iteration 4: Page 4 ===
  First doc: 23/2021/QĐ-UBND
  Found 52 new documents (total: 213)
  Navigating to: Next batch (page 5) (arg: Page$5)

=== Iteration 5: Page 5 ===
  First doc: 19/2019/QĐ-UBND (file 1)
  Found 52 new documents (total: 265)
  Navigating to: Page 6 (arg: Page$6)

=== Iteration 6: Page 6 ===
  First doc: 57/2018/QĐ-UBND
  Found 50 new documents (total: 315)
  Navigating to: Page 7 (arg: Page$7)

=== Iteration 7: Page 7 ===
  First doc: 61/2017/QĐ-UBND
  Found 50 new documents (total: 365)
  Navigating to: Page 8 (arg: P

In [24]:
import csv
import requests
import os

CSV_FILE = "documents.csv"   # file CSV chứa dữ liệu
OUTPUT_DIR = "dat_dai_law"

# Tạo thư mục nếu chưa có
os.makedirs(OUTPUT_DIR, exist_ok=True)

def sanitize_filename(name: str) -> str:
    # Loại bỏ ký tự không hợp lệ
    return "".join(c for c in name if c.isalnum() or c in "-_.")

with open(CSV_FILE, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)

    for row in reader:
        code = sanitize_filename(row["code"].replace("/", "-"))
        pdf_url = row["pdf"]

        if not pdf_url:
            print(f"Skipping {code}: missing PDF URL")
            continue

        output_path = os.path.join(OUTPUT_DIR, f"{code}.pdf")

        print(f"Downloading {code} ...")

        try:
            response = requests.get(pdf_url, stream=True, timeout=30)
            if response.status_code == 200:
                with open(output_path, "wb") as file:
                    for chunk in response.iter_content(chunk_size=8192):
                        file.write(chunk)
                print(f"Saved: {output_path}")
            else:
                print(f"Failed HTTP {response.status_code}: {pdf_url}")

        except Exception as e:
            print(f"Error downloading {pdf_url}: {e}")

Saved: dat_dai_law\292-2025-NĐ-CP.pdf
Saved: dat_dai_law\291-2025-NĐ-CP.pdf
Saved: dat_dai_law\261-2025-NĐ-CP.pdf
Saved: dat_dai_law\66.3-2025-NQ-CP.pdf
Saved: dat_dai_law\154-2025-QĐ-UBND.pdf
Saved: dat_dai_law\230-2025-NĐ-CP.pdf
Saved: dat_dai_law\226-2025-NĐ-CP.pdf
Saved: dat_dai_law\77-2025-QĐ-UBND.pdf
Saved: dat_dai_law\192-2025-NĐ-CP.pdf
Saved: dat_dai_law\50-2025-QĐ-UBND.pdf
Saved: dat_dai_law\33-2025-QĐ-UBND.pdf
Saved: dat_dai_law\32-2025-QĐ-UBND.pdf
Saved: dat_dai_law\45-2025-QĐ-UBND.pdf
Saved: dat_dai_law\43-2025-QĐ-UBND.pdf
Saved: dat_dai_law\42-2025-QĐ-UBNDfile1.pdf
Saved: dat_dai_law\42-2025-QĐ-UBNDfile2.pdf
Saved: dat_dai_law\41-2025-QĐ-UBND.pdf
Saved: dat_dai_law\201-2025-QH15.pdf
Saved: dat_dai_law\23-2025-QĐ-UBND.pdf
Saved: dat_dai_law\91-2025-NĐ-CP.pdf
Saved: dat_dai_law\28-2025-QĐ-UBNDfile1.pdf
Saved: dat_dai_law\28-2025-QĐ-UBNDfile2.pdf
Saved: dat_dai_law\28-2025-QĐ-UBNDfile3.pdf
Saved: dat_dai_law\28-2025-QĐ-UBNDfile4.pdf
Saved: dat_dai_law\75-2025-NĐ-CP.pdf
Saved:

In [27]:
import csv

CSV_FILE = "documents.csv"

total_rows = 0
pdf_count = 0
missing_pdf_count = 0
not_ubnd = 0

with open(CSV_FILE, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        total_rows += 1
        if row.get("pdf"):
            pdf_count += 1
        else:
            missing_pdf_count += 1

        if "UBND" not in row.get("code"):
            not_ubnd += 1

print(f"Total rows: {total_rows}")
print(f"Rows with PDF link: {pdf_count}")
print(f"Rows missing PDF link: {missing_pdf_count}")
print(f"Rows without UBND code: {not_ubnd}")

Total rows: 522
Rows with PDF link: 521
Rows missing PDF link: 1
Rows without UBND code: 113
